# 02 — Which model works better?

The first iteration got a complete prediction pipeline working with one model. Here we ask a different question: **if the data and validation set stay the same, which of three models predicts trip duration more accurately?** Lower RMSLE is better. This notebook is self-contained: only the competition's `train.csv` (or `train.zip`) is needed.


## 1. Load trips

One row is one taxi trip. We predict `trip_duration` in seconds. `dropoff_datetime` cannot be used because the trip has not ended when we need a prediction.


In [ ]:
from pathlib import Path

def locate_data_dir(kaggle_root=Path('/kaggle/input'), local_dir=Path('data')):
    slug = 'nyc-taxi-trip-duration'
    candidates = [kaggle_root / slug, kaggle_root / 'competitions' / slug, local_dir]
    for folder in candidates:
        has_train = (folder / 'train.csv').is_file() or (folder / 'train.zip').is_file()
        has_test = (folder / 'test.csv').is_file() or (folder / 'test.zip').is_file()
        if has_train and has_test:
            return folder
    raise FileNotFoundError('NYC Taxi train/test files not found. Attach the competition data in Kaggle Inputs.')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor

data_dir = locate_data_dir()
train_file = data_dir / 'train.csv'
if not train_file.exists():
    train_file = data_dir / 'train.zip'
train = pd.read_csv(train_file, parse_dates=['pickup_datetime'])
print('Rows, columns:', train.shape)
print('Missing values:', train.isna().sum().sum())
print('Duration in seconds:', train['trip_duration'].describe(percentiles=[.5, .99]))


## 2. Turn known trip information into features

Pickup hour and weekday may reflect traffic patterns; straight-line distance approximates how far the ride goes. It is **not** road distance. We keep the coordinates because two equally long routes can have different traffic. ID, actual drop-off time, and the answer itself are excluded.


In [ ]:
def build_features(df):
    pickup = pd.to_datetime(df['pickup_datetime'])
    lat1 = np.radians(df['pickup_latitude'].astype(float))
    lat2 = np.radians(df['dropoff_latitude'].astype(float))
    dlat = lat2 - lat1
    dlon = np.radians(df['dropoff_longitude'].astype(float) - df['pickup_longitude'].astype(float))
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    features = pd.DataFrame(index=df.index)
    features['distance_km'] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    features['pickup_hour'] = pickup.dt.hour
    features['pickup_weekday'] = pickup.dt.dayofweek
    features['pickup_month'] = pickup.dt.month
    features['pickup_latitude'] = df['pickup_latitude']
    features['pickup_longitude'] = df['pickup_longitude']
    features['dropoff_latitude'] = df['dropoff_latitude']
    features['dropoff_longitude'] = df['dropoff_longitude']
    features['passenger_count'] = df['passenger_count']
    features['vendor_id'] = df['vendor_id']
    features['store_and_fwd'] = (df['store_and_fwd_flag'] == 'Y').astype(int)
    return features

X = build_features(train)
y = train['trip_duration'].to_numpy()
assert 'trip_duration' not in X and 'dropoff_datetime' not in X and 'id' not in X
print(X.columns.tolist())


## 3. Make the comparison fair

We reserve the same 20% of trips for validation. Each model trains on the **same 150,000 rows** from the remaining 80%. We train on `log1p(duration)` because Kaggle scores with RMSLE, then convert predictions back to seconds. The baseline predicts one typical duration for every trip.


In [ ]:
def rmsle(actual, predicted):
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))

train_idx, valid_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=42)
sample_idx = np.random.default_rng(42).choice(train_idx, size=min(150_000, len(train_idx)), replace=False)
target = np.log1p(y[sample_idx])

models = {
    'Histogram Gradient Boosting': HistGradientBoostingRegressor(
        max_iter=120, max_leaf_nodes=31, learning_rate=0.08, random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=60, max_depth=20, min_samples_leaf=4,
        max_features=0.8, n_jobs=-1, random_state=42),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=60, max_depth=20, min_samples_leaf=4,
        max_features=0.8, n_jobs=-1, random_state=42),
}

baseline_seconds = np.expm1(target.mean())
results = [{'model': 'Constant baseline',
            'train_rmsle': rmsle(y[sample_idx], np.full(len(sample_idx), baseline_seconds)),
            'valid_rmsle': rmsle(y[valid_idx], np.full(len(valid_idx), baseline_seconds))}]
for name, model in models.items():
    model.fit(X.iloc[sample_idx], target)
    train_pred = np.expm1(model.predict(X.iloc[sample_idx])).clip(min=0)
    valid_pred = np.expm1(model.predict(X.iloc[valid_idx])).clip(min=0)
    results.append({'model': name,
                    'train_rmsle': rmsle(y[sample_idx], train_pred),
                    'valid_rmsle': rmsle(y[valid_idx], valid_pred)})

scores = pd.DataFrame(results)
display(scores.round(5))
best_name = scores.loc[scores['model'] != 'Constant baseline'].nsmallest(1, 'valid_rmsle')['model'].iloc[0]
print('Best on this validation split:', best_name)


## 4. Read the result, not just the winner

In our local run, validation RMSLE was **0.79661** for the baseline, **0.42925** for Histogram Gradient Boosting, **0.42277** for Random Forest, and **0.44179** for Extra Trees. Random Forest won on this split, so the end-to-end notebook used it for the next submission. However, its train RMSLE was **0.30396**: the gap to validation warns about overfitting. We have not proved it is the best model for future months or other cities.

After refitting on a 500,000-trip sample, our **late Kaggle practice submission** scored **0.41798 public / 0.41583 private**. These are not placements in the original 2017 competition. The next cell creates the submission file so you can run and submit this notebook yourself.


## 5. Create `submission.csv`

Now refit the winner on 500,000 labeled trips, predict all test trips, and save the exact `id,trip_duration` format Kaggle expects. On Kaggle the file goes to `/kaggle/working/submission.csv`. After the run finishes, use the notebook's **Submit to Competition** action to see your own score.


In [ ]:
from sklearn.base import clone

test_file = data_dir / 'test.csv'
if not test_file.exists():
    test_file = data_dir / 'test.zip'
test = pd.read_csv(test_file, parse_dates=['pickup_datetime'])
X_test = build_features(test)
assert X.columns.equals(X_test.columns)

final_idx = np.random.default_rng(42).choice(
    np.arange(len(train)), size=min(500_000, len(train)), replace=False
)
final_model = clone(models[best_name])
final_model.fit(X.iloc[final_idx], np.log1p(y[final_idx]))
predicted_seconds = np.expm1(final_model.predict(X_test)).clip(min=0)
submission = pd.DataFrame({'id': test['id'], 'trip_duration': predicted_seconds})

assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert np.isfinite(submission['trip_duration']).all()
assert submission['trip_duration'].ge(0).all()

output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output')
output_dir.mkdir(exist_ok=True)
submission_path = output_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path, 'Rows:', len(submission))
display(submission.head())
